# Near-Duplicate Text Datasets for MinHash Evaluation

## Overview

This notebook demonstrates how to load and explore the near-duplicate text datasets designed for evaluating MinHash and Jaccard similarity estimation methods.

### Dataset Description

- **Total documents**: ~163,308 across 6 datasets
- **Total duplicate pairs**: ~17,656
- **Similarity levels**: Controlled Jaccard similarities (0.1, 0.3, 0.5, 0.7, 0.9)
- **Datasets included**:
  - **Quora**: Duplicate question pairs
  - **20 Newsgroups**: Newsgroup postings
  - **MS MARCO**: Passage retrieval data
  - **C4**: Common Crawl subset
  - **AG News**: News articles
  - **Synthetic**: Controlled near-duplicate pairs

### Data Format

Each document contains:
- `input`: The raw text
- `metadata_source_dataset`: Which dataset it came from
- `metadata_duplicate_id`: ID of the duplicate pair (if applicable)
- `metadata_similarity_level`: Type of pair (duplicate, near_duplicate, different, or Jaccard level)
- `metadata_tokens`: Tokenized text (lowercased, alphanumeric only)
- `metadata_document_length`: Number of tokens


## Install Dependencies

This cell installs required packages. On Colab, most packages are pre-installed; locally, we install at matching versions.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages not pre-installed on Colab
_pip('loguru')

# Core packages (pre-installed on Colab, install locally to match)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

print("Dependencies installed successfully!")

## Imports

Import all required libraries for data loading and visualization.

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm

print("All imports successful!")

## Data Loading

Load the demo dataset from GitHub (with local fallback). The dataset contains a curated subset of examples from the full dataset.

In [ ]:
# GitHub URL for the demo data (will be available after deployment)
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-2e68d8-rateless-minhash-progressive-jaccard-est/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub load failed: {e}")
    
    # Local fallback
    if os.path.exists("mini_demo_data.json"):
        print("Loading from local file...")
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json")

# Load the data
data = load_data()
print(f"Data loaded successfully!")
print(f"Number of dataset groups: {len(data['datasets'])}")

## Explore Data Structure

Examine the structure of the loaded data and understand the format of each example.

In [ ]:
# Display dataset info
if 'dataset_info' in data:
    print("=== Dataset Info ===")
    info = data['dataset_info']
    for key, value in info.items():
        print(f"{key}: {value}")
    print()

# Explore each dataset group
for dataset_group in data['datasets']:
    dataset_name = dataset_group['dataset']
    description = dataset_group.get('description', 'No description')
    examples = dataset_group['examples']
    
    print(f"=== Dataset: {dataset_name} ===")
    print(f"Description: {description}")
    print(f"Number of examples: {len(examples)}")
    
    # Show first example structure
    if examples:
        print(f"\nFirst example keys: {list(examples[0].keys())}")
        print(f"Sample text: {examples[0]['input'][:100]}...")
        print(f"Metadata keys: {list(examples[0].keys())}")
    print()

## Flatten Data for Analysis

Convert the nested dataset structure into a flat list of examples for easier analysis.

In [ ]:
# Flatten all examples into a single list
all_examples = []
for dataset_group in data['datasets']:
    dataset_name = dataset_group['dataset']
    for example in dataset_group['examples']:
        # Add dataset name to each example for reference
        example['dataset'] = dataset_name
        all_examples.append(example)

print(f"Total examples loaded: {len(all_examples)}")

# Create a pandas DataFrame for easier analysis
df = pd.DataFrame(all_examples)
print(f"\nDataFrame shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")

## Data Statistics

Compute and display basic statistics about the dataset.

In [ ]:
# Compute statistics
print("=== Dataset Statistics ===\n")

# Count examples per dataset
dataset_counts = df['dataset'].value_counts()
print("Examples per dataset:")
print(dataset_counts)
print()

# Document length statistics
if 'metadata_document_length' in df.columns:
    lengths = df['metadata_document_length']
    print("Document length statistics:")
    print(f"  Mean: {lengths.mean():.2f}")
    print(f"  Median: {lengths.median():.2f}")
    print(f"  Min: {lengths.min()}")
    print(f"  Max: {lengths.max()}")
    print()

# Similarity level distribution
if 'metadata_similarity_level' in df.columns:
    similarity_counts = df['metadata_similarity_level'].value_counts(dropna=False)
    print("Similarity level distribution:")
    print(similarity_counts)
    print()

# Show DataFrame head
print("\n=== First 5 Examples ===")
display_cols = ['dataset', 'input', 'metadata_similarity_level', 'metadata_document_length']
display_cols = [col for col in display_cols if col in df.columns]
print(df[display_cols].head())

## Visualize Data

Create visualizations to better understand the dataset characteristics.

In [ ]:
# Set up the figure
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Dataset Exploration Visualizations', fontsize=14)

# Plot 1: Examples per dataset
ax1 = axes[0]
dataset_counts.plot(kind='bar', ax=ax1, color='skyblue')
ax1.set_title('Examples per Dataset')
ax1.set_xlabel('Dataset')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=45)

# Plot 2: Document length distribution
ax2 = axes[1]
if 'metadata_document_length' in df.columns:
    lengths = df['metadata_document_length']
    ax2.hist(lengths, bins=20, color='lightgreen', edgecolor='black', alpha=0.7)
    ax2.set_title('Document Length Distribution')
    ax2.set_xlabel('Number of Tokens')
    ax2.set_ylabel('Frequency')
    ax2.axvline(lengths.mean(), color='red', linestyle='--', label=f'Mean: {lengths.mean():.1f}')
    ax2.legend()

# Plot 3: Similarity level distribution
ax3 = axes[2]
if 'metadata_similarity_level' in df.columns:
    similarity_counts = df['metadata_similarity_level'].value_counts(dropna=False)
    # Replace NaN with 'None' for display
    labels = [str(x) if pd.notna(x) else 'None' for x in similarity_counts.index]
    ax3.pie(similarity_counts.values, labels=labels, autopct='%1.1f%%', startangle=90)
    ax3.set_title('Similarity Level Distribution')

plt.tight_layout()
plt.show()

## Token Analysis

Analyze the tokenized text to understand vocabulary and token patterns.

In [ ]:
# Analyze tokens
print("=== Token Analysis ===\n")

# Collect all tokens
all_tokens = []
for example in all_examples:
    if 'metadata_tokens' in example:
        all_tokens.extend(example['metadata_tokens'])

print(f"Total tokens across all examples: {len(all_tokens)}")
print(f"Unique tokens: {len(set(all_tokens))}")
print()

# Most common tokens
token_counts = Counter(all_tokens)
print("Top 20 most common tokens:")
for token, count in token_counts.most_common(20):
    print(f"  '{token}': {count}")
print()

# Average tokens per document
if 'metadata_tokens' in df.columns:
    avg_tokens = df['metadata_tokens'].apply(len).mean()
    print(f"Average tokens per document: {avg_tokens:.2f}")

## Example: Jaccard Similarity Calculation

Demonstrate how to compute Jaccard similarity between document pairs using token sets.

In [ ]:
def compute_jaccard_similarity(tokens1, tokens2):
    """Compute Jaccard similarity between two token sets."""
    set1 = set(tokens1)
    set2 = set(tokens2)
    
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    
    if union == 0:
        return 0.0
    return intersection / union

# Find duplicate pairs and compute their Jaccard similarity
print("=== Jaccard Similarity Examples ===\n")

# Group examples by duplicate_id
duplicate_pairs = {}
for example in all_examples:
    dup_id = example.get('metadata_duplicate_id')
    if dup_id and pd.notna(dup_id):
        if dup_id not in duplicate_pairs:
            duplicate_pairs[dup_id] = []
        duplicate_pairs[dup_id].append(example)

# Display some duplicate pairs and their Jaccard similarity
for dup_id, examples in list(duplicate_pairs.items())[:3]:
    if len(examples) >= 2:
        tokens1 = examples[0].get('metadata_tokens', [])
        tokens2 = examples[1].get('metadata_tokens', [])
        jaccard = compute_jaccard_similarity(tokens1, tokens2)
        
        print(f"Duplicate ID: {dup_id}")
        print(f"  Text 1: {examples[0]['input'][:80]}...")
        print(f"  Text 2: {examples[1]['input'][:80]}...")
        print(f"  Jaccard Similarity: {jaccard:.4f}")
        print(f"  Similarity Level: {examples[0].get('metadata_similarity_level', 'N/A')}")
        print()

if not duplicate_pairs:
    print("No duplicate pairs found in this small demo subset.")
    print("The full dataset contains 17,656 duplicate pairs.")

## Summary

This notebook demonstrated:

1. **Loading the dataset** from GitHub with local fallback
2. **Exploring the data structure** - understanding the format and metadata
3. **Computing statistics** - dataset sizes, document lengths, similarity levels
4. **Visualizing the data** - bar charts, histograms, and pie charts
5. **Analyzing tokens** - vocabulary size and most common tokens
6. **Computing Jaccard similarity** - the core metric for near-duplicate detection

### Next Steps

With the full dataset (163K documents), you can:
- Evaluate MinHash algorithms for near-duplicate detection
- Benchmark Jaccard similarity estimation methods
- Test performance across different similarity levels
- Compare results across the 6 different dataset types
